In [1]:
from transformers.models.llama.modeling_llama import LlamaForCausalLM, LlamaConfig
from torch import nn
from typing import *
class LlamaForRankCausalLM(LlamaForCausalLM):

    def __init__(
        self, 
        config: "LlamaConfig", 
        rel_score_id: int = 32002, 
        verify_score_id: int = 2, 
        beta: float = 0.5, 
        rr_loss_fn : str = "bce", 
        num_labels: int = 1, 
        enable_verify: bool = False, 
        bias: float = 0.4, 
        smoothing: bool = False,
        psg_num: int = 4, 
        rank_only: bool = False,
        relevant_init: Union[bool, str, int] = None,
        ):
        super().__init__(config)

        self.num_labels = num_labels
        self.beta = beta
        
        self.rel_score = nn.Linear(config.hidden_size, num_labels, bias=False)
        self.rel_score_id = rel_score_id

        self.list_wise = False
        self.rank_only = rank_only
        self.enable_verify = enable_verify
        
        if relevant_init:
            if isinstance(relevant_init, str):
                if relevant_init == "relevant":
                    weight_1 = self.lm_head.weight[8018: 8019].detach().clone()
                    weight_2 = self.lm_head.weight[28190: 28191].detach().clone()
                elif relevant_init == "useful":
                    weight_1 = self.lm_head.weight[5407: 5408].detach().clone()
                    weight_2 = self.lm_head.weight[19315: 19316].detach().clone()
                else:
                    raise NotImplementedError
                self.rel_score.weight = nn.Parameter(weight_1 - weight_2)
            elif isinstance(relevant_init, int):
                weight = self.lm_head.weight.data[relevant_init: relevant_init + 1].detach().clone()
                self.rel_score.weight = nn.Parameter(weight)
            else:
                raise NotImplementedError
        self.post_init()      

[2024-01-02 18:45:05,060] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [5]:
model = LlamaForRankCausalLM.from_pretrained("/home/wangyuhao/self-rag/Llama-2-7b-hf", device_map='cpu', relevant_init="useful")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForRankCausalLM were not initialized from the model checkpoint at /home/wangyuhao/self-rag/Llama-2-7b-hf and are newly initialized: ['rel_score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/wangyuhao/miniconda3/envs/llm/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:362: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/wangyuhao/miniconda3/envs/llm/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:367: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in s

In [3]:
model.rel_score.weight

Parameter containing:
tensor([[ 0.0047, -0.0249,  0.0135,  ..., -0.0195,  0.0147,  0.0217]],
       requires_grad=True)

In [4]:
weight_1 = model.lm_head.weight[5407: 5408].detach().clone()
weight_2 = model.lm_head.weight[19315: 19316].detach().clone()
weight_1 - weight_2

tensor([[-0.0042,  0.0163,  0.0013,  ..., -0.0103,  0.0034,  0.0273]])

In [7]:
from peft import LoraConfig, TaskType, get_peft_model
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=1,
    lora_dropout=0,
    target_modules=["q_proj"],
    modules_to_save=["rel_score"],
)

In [6]:
peft_model = get_peft_model(model, lora_config)

In [11]:
# peft_model.rel_score.modules_to_save.default

# if relevant_init == "relevant":
weight_1 = model.lm_head.weight[8018: 8019].detach().clone()
weight_2 = model.lm_head.weight[28190: 28191].detach().clone()
model.rel_score.weight = nn.Parameter(weight_1 - weight_2)
model.save_pretrained("/mnt/wangyuhao/usere/relevant-sub", safe_serialization=True)

In [14]:
# elif relevant_init == "useful":
weight_1 = model.lm_head.weight[5407: 5408].detach().clone()
weight_2 = model.lm_head.weight[19315: 19316].detach().clone()
peft_model.rel_score.modules_to_save.default.weight = nn.Parameter(weight_1 - weight_2)
peft_model.save_pretrained("/mnt/wangyuhao/usere/useful-sub", safe_serialization=True)

In [15]:
weight = model.lm_head.weight[8018: 8019].detach().clone()
peft_model.rel_score.modules_to_save.default.weight = nn.Parameter(weight)
peft_model.save_pretrained("/mnt/wangyuhao/usere/relevant", safe_serialization=True)

weight = model.lm_head.weight[5407: 5408].detach().clone()
peft_model.rel_score.modules_to_save.default.weight = nn.Parameter(weight)
peft_model.save_pretrained("/mnt/wangyuhao/usere/useful", safe_serialization=True)

In [6]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, "/mnt/wangyuhao/usere/useful")
model.merge_and_unload()
type(model)

peft.peft_model.PeftModelForCausalLM

In [4]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForRankCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
              (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
              (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
              (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
              (rotary_emb): LlamaRotaryEmbedding()
            )
            (mlp): LlamaMLP(
              (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
              (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
              (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
              (act_fn): SiLUActivation()
         

In [ ]:
model.save_pretrained("/mnt/wangyuhao/usere/useful")

In [6]:
model.merge_and_unload()
type(model)

peft.peft_model.PeftModelForCausalLM

In [5]:
type(model)

peft.peft_model.PeftModelForCausalLM

In [21]:
model.lm_head.weight[5407: 5408].detach().clone()

tensor([[-0.0052,  0.0118, -0.0040,  ...,  0.0013, -0.0046,  0.0222]])